# 04 · Detección de Outliers Multivariados por Votación (Consenso)
## Módulo 3 — Feature Engineering & Modelado Supervisado

---

### 🎯 Objetivo del Cuaderno
Identificar y tratar observaciones atípicas multivariadas en el conjunto de entrenamiento ($X_{train}$) mediante un **Sistema de Votación por Consenso** entre 3 detectores con principios matemáticos distintos:
1. **Isolation Forest (Aislamiento mediante Árboles):** Mide la facilidad con la que una observación queda aislada en cortes aleatorios.
2. **Local Outlier Factor - LOF (Densidad Local):** Compara la densidad local de un registro con la de sus $k$ vecinos más cercanos.
3. **Elliptic Envelope / Covarianza Robusta (Geometría de Mahalanobis):** Evalúa la distancia espacial asumiendo una envolvente elíptica sobre la masa central de datos.

---

### 🧠 Construcción de la Matriz Numérica Inteligente (Preservación de Jerarquía)
Para que los algoritmos calculen distancias sin perder la escala ordinal natural de los datos socioeconómicos, aplicamos una transformación mixta temporal sobre $X_{train}$:
* **Variables Ordinales (`age`, `household_income`, `years_on_internet`):** Mapeadas mediante valores enteros secuenciales ($1, 2, 3, \dots$) para reflejar jerarquía de magnitud.
* **Variables Nominales (`primary_computing_platform`, `clean_country`, etc.):** Codificadas mediante *One-Hot Encoding* (Dummies).
* **Escalado Estándar (`StandardScaler`):** Normalización de la matriz continua para igualar la influencia espacial de todas las dimensiones.

🔒 **Regla Metodológica de No Contaminación (*Train-Only*):**
* En **TRAIN ($X_{train}$)**: Las observaciones clasificadas como outliers por consenso ($\ge 2$ votos) **se eliminan** para evitar que vicien el aprendizaje de los modelos.
* En **TEST ($X_{test}$)**: **Se conservan el 100% de las filas**, generando únicamente un flag o puntuación diagnóstica de alerta.

In [1]:
# ==============================================================================
# BLOQUE 1: CARGA Y CONSTRUCCIÓN DE LA MATRIZ NUMÉRICA INTELIGENTE
# ==============================================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# 1. Carga de artefactos limpios del Módulo 03
ruta_train = 'data_processed/03_train_ecommerce.csv'
ruta_test = 'data_processed/03_test_ecommerce.csv'

train_df = pd.read_csv(ruta_train)
test_df = pd.read_csv(ruta_test)

target_col = 'Cluster'

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print(f"📦 Datos de Train cargados: {X_train.shape[0]} filas × {X_train.shape[1]} columnas.")

# ------------------------------------------------------------------------------
# 2. MAPEO ORDINAL MANUAL (Jerarquía socioeconómica y tecnológica)
# ------------------------------------------------------------------------------
# Definimos los mapas respetando la secuencia lógica
mapa_age = {
    '< 18 años': 1, '18 - 24 años': 2, '25 - 34 años': 3, 
    '35 - 44 años': 4, '45 - 54 años': 5, '55 - 64 años': 6, '65+ años': 7
}

mapa_income = {
    '< $10k': 1, '$10k - $24k': 2, '$25k - $49k': 3, 
    '$50k - $74k': 4, '$75k - $99k': 5, '> $100k': 6
}

mapa_internet = {
    '0 - 0.5 años': 1, '0.5 - 1 año': 2, '1 - 3 años': 3, 
    '3 - 5 años': 4, '5 - 7 años': 5, '> 7 años': 6
}

def crear_matriz_numerica_inteligente(df):
    df_num = df.copy()
    
    # Aplicamos mapeo ordinal en las columnas donde aplique (rellenando no mapeados con 0)
    if 'age' in df_num.columns:
        df_num['age_num'] = df_num['age'].map(mapa_age).fillna(0)
        df_num = df_num.drop(columns=['age'])
        
    if 'household_income' in df_num.columns:
        df_num['income_num'] = df_num['household_income'].map(mapa_income).fillna(0)
        df_num = df_num.drop(columns=['household_income'])
        
    if 'years_on_internet' in df_num.columns:
        df_num['internet_num'] = df_num['years_on_internet'].map(mapa_internet).fillna(0)
        df_num = df_num.drop(columns=['years_on_internet'])
        
    # One-Hot Encoding para las variables nominales restantes
    df_encoded = pd.get_dummies(df_num, drop_first=True, dtype=float)
    return df_encoded

# Construimos la matriz espacial para Train
X_train_encoded = crear_matriz_numerica_inteligente(X_train)

# Escalado Estándar para igualar influencias de distancia
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)

print(f"📐 Matriz Espacial Inteligente construida: {X_train_scaled.shape[0]} observaciones × {X_train_scaled.shape[1]} dimensiones continuas.")

📦 Datos de Train cargados: 8086 filas × 61 columnas.
📐 Matriz Espacial Inteligente construida: 8086 observaciones × 169 dimensiones continuas.


## 2 y 3. Entrenamiento del Jurado de Detectores y Votación por Consenso

Entrenamos tres algoritmos no supervisados sobre la Matriz Espacial Inteligente ($X_{train\_scaled}$):

1. **Isolation Forest (Basado en Aislamiento):** Aísla observaciones mediante árboles aleatorios (`contamination=0.03`).
2. **Local Outlier Factor - LOF (Basado en Densidad Local):** Compara la densidad de k-vecinos con `novelty=True`.
3. **Elliptic Envelope (Geometría y Covarianza Robusta):** Modela la envolvente elíptica central de los datos. *(Para evitar problemas de colinealidad geométrica por las variables dummy, reduce previamente la dimensionalidad a componentes principales PCA)*.

### 🗳️ Regla de Consenso:
Una observación se declara oficialmente como **Outlier Multivariado** si al menos 2 de los 3 modelos votan que es un registro anómalo ($\text{Votos} \ge 2$).

In [3]:
# ==============================================================================
# BLOQUE 2 Y 3: JURADO DE DETECTORES Y SISTEMA DE VOTACIÓN POR CONSENSO
# ==============================================================================
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope
from sklearn.decomposition import PCA

# Fijamos una tasa esperada de contaminación prudente (3%)
CONTAMINACION = 0.03

print("🤖 1. ENTRENANDO DETECTOR 1: Isolation Forest...")
iso_forest = IsolationForest(contamination=CONTAMINACION, random_state=42, n_jobs=-1)
pred_iso = iso_forest.fit_predict(X_train_scaled)
# Convertimos salida (-1 outlier, 1 normal) a binario (1 outlier, 0 normal)
voto_iso = np.where(pred_iso == -1, 1, 0)

print("🤖 2. ENTRENANDO DETECTOR 2: Local Outlier Factor (LOF)...")
lof = LocalOutlierFactor(n_neighbors=20, contamination=CONTAMINACION, novelty=True, n_jobs=-1)
lof.fit(X_train_scaled)
pred_lof = lof.predict(X_train_scaled)
voto_lof = np.where(pred_lof == -1, 1, 0)

print("🤖 3. ENTRENANDO DETECTOR 3: Elliptic Envelope (Mahalanobis / MCD)...")
# Usamos PCA (90% varianza) para estabilizar la matriz de covarianza antes de la elipse
pca_env = PCA(n_components=0.90, random_state=42)
X_train_pca = pca_env.fit_transform(X_train_scaled)

elliptic = EllipticEnvelope(contamination=CONTAMINACION, random_state=42)
elliptic.fit(X_train_pca)
pred_elliptic = elliptic.predict(X_train_pca)
voto_elliptic = np.where(pred_elliptic == -1, 1, 0)

# ------------------------------------------------------------------------------
# MATRIZ DE VOTACIÓN Y CONSENSO
# ------------------------------------------------------------------------------
df_votos = pd.DataFrame({
    'IsoForest': voto_iso,
    'LOF': voto_lof,
    'EllipticEnv': voto_elliptic
}, index=X_train.index)

# Sumamos los votos por fila (rango de 0 a 3)
df_votos['Total_Votos'] = df_votos.sum(axis=1)

# Marcamos como outlier de consenso si tiene 2 o más votos
df_votos['Outlier_Consenso'] = np.where(df_votos['Total_Votos'] >= 2, 1, 0)

print("\n" + "="*75)
print("🗳️ RESUMEN DE DETECCIONES INDIVIDUALES Y VOTOS:")
print(f"· Isolation Forest detectó:    {voto_iso.sum()} outliers ({voto_iso.mean()*100:.2f}%)")
print(f"· LOF detectó:                 {voto_lof.sum()} outliers ({voto_lof.mean()*100:.2f}%)")
print(f"· Elliptic Envelope detectó:   {voto_elliptic.sum()} outliers ({voto_elliptic.mean()*100:.2f}%)")
print("-" * 75)
print("📊 DISTRIBUCIÓN DEL CONSEJO DE VOTACIÓN (Total de Votos):")
display(df_votos['Total_Votos'].value_counts().sort_index().to_frame('Cantidad_Observaciones'))

outliers_detectados = df_votos[df_votos['Outlier_Consenso'] == 1].index
print(f"\n🚨 OUTLIERS CONFIRMADOS POR CONSENSO (≥ 2 VOTOS): {len(outliers_detectados)} filas ({len(outliers_detectados)/len(X_train)*100:.2f}% de Train)")

🤖 1. ENTRENANDO DETECTOR 1: Isolation Forest...
🤖 2. ENTRENANDO DETECTOR 2: Local Outlier Factor (LOF)...
🤖 3. ENTRENANDO DETECTOR 3: Elliptic Envelope (Mahalanobis / MCD)...

🗳️ RESUMEN DE DETECCIONES INDIVIDUALES Y VOTOS:
· Isolation Forest detectó:    243 outliers (3.01%)
· LOF detectó:                 178 outliers (2.20%)
· Elliptic Envelope detectó:   243 outliers (3.01%)
---------------------------------------------------------------------------
📊 DISTRIBUCIÓN DEL CONSEJO DE VOTACIÓN (Total de Votos):


,Cantidad_Observaciones
Total_Votos,
0,7498
1,520
2,60
3,8



🚨 OUTLIERS CONFIRMADOS POR CONSENSO (≥ 2 VOTOS): 68 filas (0.84% de Train)


## 4. Perfilamiento y Depuración del Conjunto de Entrenamiento

Aplicamos el protocolo estricto de gestión de outliers para el modelado supervisado:

1. **En TRAIN ($X_{train}$):** Eliminamos las $68$ observaciones confirmadas por consenso ($\ge 2$ votos). Al removerlos de la muestra de entrenamiento, evitamos que los algoritmos de clasificación (árboles, regresiones o redes) distorsionen sus fronteras de decisión tratando de adaptarse a registros con combinaciones incoherentes.
2. **En TEST ($X_{test}$):** Conservamos la muestra de prueba **$100\%$ intacta**. En producción, los modelos deben evaluarse frente a la distribución real sin alterar el tamaño de la muestra de prueba.

In [4]:
# ==============================================================================
# BLOQUE 4 Y 5: DEPURACIÓN DE TRAIN Y EXPORTACIÓN DE ARTEFACTOS FINALES
# ==============================================================================
import json
import os

# 1. Identificamos las observaciones normales (menos de 2 votos) en Train
mask_train_validos = (df_votos['Outlier_Consenso'] == 0)

# 2. Depuramos Train
train_04 = train_df[mask_train_validos].copy()

# Test se mantiene intacto (conserva todas sus filas)
test_04 = test_df.copy()

print(f"✂️ Filas eliminadas de Train por consenso multivariado: {len(outliers_detectados)}")
print(f"📊 Train original: {len(train_df)} filas  ➡️  Train depurado: {len(train_04)} filas")
print(f"🔒 Test conservado intacto: {len(test_04)} filas")

# ------------------------------------------------------------------------------
# 3. GUARDADO DE BASES PROCESADAS Y REPORTE DE OUTLIERS
# ------------------------------------------------------------------------------
os.makedirs('data_processed', exist_ok=True)
os.makedirs('reports', exist_ok=True)

train_04.to_csv('data_processed/04_train_ecommerce.csv', index=False)
test_04.to_csv('data_processed/04_test_ecommerce.csv', index=False)

reporte_outliers = {
    'total_registros_train_inicial': len(train_df),
    'detecciones_individuales': {
        'isolation_forest': int(voto_iso.sum()),
        'local_outlier_factor': int(voto_lof.sum()),
        'elliptic_envelope': int(voto_elliptic.sum())
    },
    'outliers_consenso_eliminados': len(outliers_detectados),
    'pct_outliers_eliminados': round(float(len(outliers_detectados)/len(train_df)*100), 2),
    'total_registros_train_depurado': len(train_04)
}

with open('reports/outliers.json', 'w') as f:
    json.dump(reporte_outliers, f, indent=4)

print("="*75)
print("✅ ARTEFACTOS DEL MÓDULO 04 EXPORTADOS CON ÉXITO:")
print(f"· Train Módulo 04 guardado: 'data_processed/04_train_ecommerce.csv' ({train_04.shape[0]} filas)")
print(f"· Test Módulo 04 guardado:  'data_processed/04_test_ecommerce.csv' ({test_04.shape[0]} filas)")
print("· Reporte JSON guardado en: 'reports/outliers.json'")
print("="*75)

✂️ Filas eliminadas de Train por consenso multivariado: 68
📊 Train original: 8086 filas  ➡️  Train depurado: 8018 filas
🔒 Test conservado intacto: 2022 filas
✅ ARTEFACTOS DEL MÓDULO 04 EXPORTADOS CON ÉXITO:
· Train Módulo 04 guardado: 'data_processed/04_train_ecommerce.csv' (8018 filas)
· Test Módulo 04 guardado:  'data_processed/04_test_ecommerce.csv' (2022 filas)
· Reporte JSON guardado en: 'reports/outliers.json'


## 5. Conclusiones de la Detección de Outliers Multivariados

El proceso de diagnóstico espacial mediante votación por consenso arroja las siguientes conclusiones metodológicas y de negocio:

---

### A. Eficiencia del Sistema de Votación (Consenso)
* **Filtrado Prudente y Preciso:** La votación exigió la coincidencia de al menos dos algoritmos con fundamentos distintos (aislamiento por árboles, densidad local y geometría de Mahalanobis). Esto previno el descarte masivo de datos y aisló únicamente **$68$ observaciones ($0.84\%$ de $Train$)** con combinaciones verdaderamente atípicas o contradictorias.
* **Preservación de la Matriz de Atributos:** La eliminación de filas se realizó sobre la base categórica original limpia, utilizando la Matriz Numérica Inteligente (ordinal + una-hot) de $169$ dimensiones únicamente como espacio de cálculo de distancias.

---

### B. Mantenimiento del Rigor Metodológico (*Train-Only*)
* **Depuración sobre $Train$:** Remover estos $68$ registros de $X_{train}$ estabiliza la estimación de parámetros en las fases posteriores de *Weight of Evidence (WoE)* e Ingeniería de Características.
* **Integridad del $Test$:** El conjunto $X_{test}$ no sufrió eliminación de observaciones, garantizando que el modelo supervisado sea evaluado sobre la distribución real completa.

---

### 🏁 Transición al Siguiente Módulo:
Con un conjunto de entrenamiento limpio, sin variables inútiles y libre de anomalías espaciales, se procede al cuaderno **`05_woe_iv.ipynb`**, donde mediremos la capacidad predictiva individual de cada categoría mediante **Weight of Evidence (WoE)** e **Information Value (IV)**.